In [1]:
print("hello")

hello


In [2]:
import os
%pwd

'd:\\MLOps Udemy Krish Naik\\Text-Summarizer\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\MLOps Udemy Krish Naik\\Text-Summarizer'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvalConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_path: Path

In [6]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import read_yaml,create_dir
from src.text_summarizer.exception.exception import SummaryException
import sys

In [9]:
class ConfigurationManager:
    def __init__(self,config_path=CONFIG_FILE_PATH,params_file_path=PARAMS_FILE_PATH):
        self.config = read_yaml(path_to_yaml=config_path)
        self.params = read_yaml(path_to_yaml=params_file_path)

        create_dir([self.config.artifacts_root])

    def get_model_eval_config(self)-> ModelEvalConfig:
        config = self.config.model_eval

        create_dir([config.root_dir])

        model_eval_config = ModelEvalConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path=config.model_path,
            tokenizer_path=config.tokenizer_path,
            metric_file_path=config.metric_file_path
        )
        return model_eval_config

In [10]:
import torch
from datasets import load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq
from tqdm import tqdm
import pandas as pd

In [11]:
import evaluate


class ModelEval:
    def __init__(self,config:ModelEvalConfig):
        self.config = config

    def generate_batch_sized_chunks(self,list_of_elements, batch_size):
        """split the dataset into smaller batches that we can process simultaneously
        Yield successive batch-sized chunks from list_of_elements."""

        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i : i + batch_size]

    def calculate_metric_on_test_ds(self,dataset, metric, model, tokenizer,batch_size=16, 
                                    device="cuda" if torch.cuda.is_available() else "cpu",column_text="article",column_summary="highlights"):
        article_batches = list(self.generate_batch_sized_chunks(dataset[column_text], batch_size))
        target_batches = list(self.generate_batch_sized_chunks(dataset[column_summary], batch_size))
        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches), total=len(article_batches)):
            inputs = tokenizer(article_batch, max_length=1024,  truncation=True,padding="max_length", return_tensors="pt")

            summaries = model.generate(input_ids=inputs["input_ids"].to(device),
                            attention_mask=inputs["attention_mask"].to(device),
                            length_penalty=0.8, num_beams=8, max_length=128)

            decoded_summaries = [tokenizer.decode(s, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
                for s in summaries]

            decoded_summaries = [d.replace("", " ") for d in decoded_summaries]
            metric.add_batch(predictions=decoded_summaries, references=target_batch)
        
        score = metric.compute()
        print(self.config.data_path)
        return score
    
    def eval(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        tokenizer = AutoTokenizer.from_pretrained(self.config.tokenizer_path)
        model_peg = AutoModelForSeq2SeqLM.from_pretrained(self.config.model_path).to(device)

        #load data

        data_sam_pt = load_from_disk(self.config.data_path)
        rouge_names = ["rouge1","rouge2","rougeL","rougeLsum"]

        rouge_metric = evaluate.load('rouge')

        score = self.calculate_metric_on_test_ds(
            data_sam_pt['test'][0:10], rouge_metric,  model_peg, tokenizer, batch_size = 2, column_text = 'dialogue', column_summary= 'summary'
        )
        rouge_dict = {rn: score[rn] for rn in rouge_names }


        df = pd.DataFrame(rouge_dict, index = [f'pegasus'] )
        df.to_csv(self.config.metric_file_path)

In [12]:
config = ConfigurationManager()
model_eval_config = config.get_model_eval_config()
model_eval = ModelEval(config=model_eval_config)
model_eval.eval()

100%|██████████| 5/5 [03:19<00:00, 39.83s/it]


artifacts/data_transformation/samsung_dataset


In [22]:
data_sam_pt = load_from_disk("artifacts/data_transformation/samsung_dataset")
data_sam_pt['test'][0:10]

{'id': ['13862856',
  '13729565',
  '13680171',
  '13729438',
  '13828600',
  '13716964',
  '13731487',
  '13814882',
  '13680876',
  '13809974'],
 'dialogue': ["Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
  "Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric: I know! And shows how Americans see Russian ;)\r\nRob: And it's really funny!\r\nEric: I know! I especially like the train part!\r\nRob: Hahaha! No one talks to the machine like that!\r\nEric: Is this his only stand-up?\r\nRob: Idk. I'll check.\r\nEric: Sure.\r\nRob: Turns out no! There are some of his stand-ups on youtube.\r\nEric: Gr8! I'll watch them now!\r\n